# Gerador de Letras com LLM (BLOOM 1.7B + LoRA)

Fine-tune eficiente de BLOOM-1b7 no corpus de letras brasileiras (~5.159 musicas).

## Por que BLOOM + LoRA?

| | GPT-2 PT (antes) | BLOOM-1b7 + LoRA (agora) |
|---|---|---|
| Parametros | 124M | 1.700M |
| Treino | Full fine-tune (124M params) | LoRA (~4M params, 0.2%) |
| Portugues | Razoavel | Bom (treinado em corpus PT) |
| Coerencia | Baixa | Alta |
| VRAM | ~3 GB | ~7 GB (cabe na T4) |

**LoRA** (Low-Rank Adaptation) treina apenas adaptadores pequenos sobre o modelo grande congelado. Isso preserva todo o conhecimento linguistico do BLOOM enquanto ensina o estilo das letras.

## Como usar

1. Conecte GPU: `Runtime > Change runtime type > T4 GPU`
2. Faca upload do `training_corpus.txt` para `Google Drive > musicas_model/`
3. Execute as celulas em ordem
4. Primeira vez: treine (~20-40 min na T4)
5. Proximas vezes: carregue direto do Drive (secao 8)

---

## 1. Setup do Ambiente

In [ ]:
# Verificar GPU
!nvidia-smi

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ATENCAO: GPU nao detectada! Ative em Runtime > Change runtime type > T4 GPU")

In [ ]:
# Instalar dependencias
!pip install -q transformers datasets accelerate peft
print("Dependencias instaladas.")

In [ ]:
import os
import re
import json
import random
from collections import Counter

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

set_seed(42)
print("Imports OK.")

## 2. Montar Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/musicas_model'
os.makedirs(DRIVE_BASE, exist_ok=True)

CORPUS_PATH = os.path.join(DRIVE_BASE, 'training_corpus.txt')
ADAPTER_SAVE_DIR = os.path.join(DRIVE_BASE, 'bloom-letras-lora')

print(f"Corpus: {CORPUS_PATH}")
print(f"Adapter LoRA: {ADAPTER_SAVE_DIR}")

if os.path.exists(CORPUS_PATH):
    size_mb = os.path.getsize(CORPUS_PATH) / (1024 * 1024)
    print(f"\nCorpus encontrado! ({size_mb:.1f} MB)")
else:
    print(f"\nCorpus NAO encontrado em {CORPUS_PATH}")
    print("Faca upload do training_corpus.txt para musicas_model/ no Drive.")

## 3. Preparar Corpus

Formato de treino com linguagem natural (sem tokens especiais):
```
[Genero: Pagode]

Primeira linha da letra
Segunda linha...

[FIM]
```

BLOOM ja entende portugues, entao prompts em linguagem natural funcionam melhor que tokens artificiais.

In [ ]:
def parse_training_corpus(filepath):
    """Le o training_corpus.txt e retorna lista de dicts."""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    raw_songs = content.split('=' * 80)
    songs = []
    header_pattern = re.compile(r'###\s+(.+?)\s+-\s+(.+?)\s+\[(.+?)\]')

    for raw in raw_songs:
        raw = raw.strip()
        if not raw:
            continue

        lines = raw.split('\n')
        match = header_pattern.match(lines[0].strip())
        if match:
            title = match.group(1).strip()
            artist = match.group(2).strip()
            genre = match.group(3).strip()
            lyrics = '\n'.join(lines[1:]).strip()
        else:
            title, artist, genre = 'Desconhecida', 'Desconhecido', 'MPB'
            lyrics = raw.strip()

        if len(lyrics) > 50:
            songs.append({'title': title, 'artist': artist, 'genre': genre, 'lyrics': lyrics})

    return songs


songs = parse_training_corpus(CORPUS_PATH)
print(f"Musicas carregadas: {len(songs)}")

genre_counts = Counter(s['genre'] for s in songs)
print("\nDistribuicao por genero:")
for genre, count in genre_counts.most_common():
    print(f"  {genre:<12} {count:>5}")

In [ ]:
def format_song(song):
    """
    Formata para treino com prompts em linguagem natural.
    BLOOM entende melhor texto natural do que tokens especiais.
    """
    genre = song['genre']
    lyrics = song['lyrics'].strip()
    return f"[Genero: {genre}]\n\n{lyrics}\n\n[FIM]"


formatted_texts = [format_song(s) for s in songs]

print(f"Musicas formatadas: {len(formatted_texts)}")
print(f"\nExemplo:")
print(formatted_texts[0][:400])
print("...")

## 4. Carregar BLOOM-1b7 + Configurar LoRA

**BLOOM-1b7**: Modelo de 1.7 bilhoes de parametros treinado em 46 idiomas incluindo portugues.

**LoRA** (r=16): Adiciona ~4M parametros treinaveis (0.2% do total). O modelo base fica congelado - so os adaptadores aprendem o estilo das letras.

In [ ]:
BASE_MODEL = 'bigscience/bloom-1b7'

print(f"Carregando tokenizer de {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# BLOOM ja tem pad_token, mas vamos garantir
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocabulario: {len(tokenizer)} tokens")
print(f"Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

In [ ]:
print(f"Carregando modelo {BASE_MODEL} em fp16...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
)

# Configurar LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                              # Rank dos adaptadores
    lora_alpha=32,                     # Escala do LoRA
    lora_dropout=0.05,                 # Dropout para regularizacao
    target_modules=["query_key_value"],  # Camadas de atencao do BLOOM
    bias="none",
)

model = get_peft_model(model, lora_config)

# Mostrar parametros
model.print_trainable_parameters()

# Verificar VRAM
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM: {vram_used:.1f} / {vram_total:.1f} GB")

## 5. Criar Dataset de Treino

In [ ]:
class LyricsDataset(Dataset):
    """Dataset de letras tokenizadas."""

    def __init__(self, texts, tokenizer, max_length=512):
        self.examples = []
        print(f"Tokenizando {len(texts)} musicas (max_length={max_length})...")

        too_long = 0
        for text in texts:
            encoding = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding='max_length',
                return_tensors='pt',
            )
            input_ids = encoding['input_ids'].squeeze()
            attention_mask = encoding['attention_mask'].squeeze()

            # Labels: -100 nos tokens de padding para ignorar no loss
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100

            if len(tokenizer.encode(text)) > max_length:
                too_long += 1

            self.examples.append({
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
            })

        print(f"Dataset: {len(self.examples)} exemplos")
        if too_long > 0:
            print(f"  ({too_long} truncadas para {max_length} tokens)")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


# Split treino/validacao 95/5
random.shuffle(formatted_texts)
split_idx = int(len(formatted_texts) * 0.95)
train_texts = formatted_texts[:split_idx]
val_texts = formatted_texts[split_idx:]

MAX_LENGTH = 512
train_dataset = LyricsDataset(train_texts, tokenizer, max_length=MAX_LENGTH)
val_dataset = LyricsDataset(val_texts, tokenizer, max_length=MAX_LENGTH)

print(f"\nTreino: {len(train_dataset)} | Validacao: {len(val_dataset)}")

## 6. Treinar (LoRA)

Com LoRA treinamos apenas ~4M parametros (0.2%), o que:
- Treina **muito mais rapido** que full fine-tune
- **Nao destroi** o conhecimento linguistico do BLOOM
- **Menos overfitting** - o modelo generaliza melhor

**Tempo estimado: 20-40 min na T4**

In [ ]:
OUTPUT_DIR = '/content/bloom-letras-checkpoints'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Hiperparametros (ajustados para LoRA)
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # Batch efetivo = 16
    learning_rate=2e-4,              # Maior que full fine-tune (padrao para LoRA)
    weight_decay=0.01,
    warmup_steps=50,
    lr_scheduler_type='cosine',

    # Avaliacao e salvamento
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    # Performance
    fp16=True,
    dataloader_num_workers=2,
    gradient_checkpointing=True,     # Economiza VRAM

    # Logging
    logging_steps=25,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Trainer configurado.")
print(f"  Batch efetivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Gradient checkpointing: {training_args.gradient_checkpointing}")

In [ ]:
# ====================================================================
# TREINAR - Execute apenas na primeira vez.
# Nas proximas vezes, pule para a secao 8 (Carregar do Drive).
# ====================================================================

print("Iniciando treino LoRA...")
print("(~20-40 min na T4)\n")

train_result = trainer.train()

print("\n" + "=" * 60)
print(" TREINO FINALIZADO ".center(60, "="))
print("=" * 60)
print(f"  Loss final: {train_result.training_loss:.4f}")
print(f"  Steps: {train_result.global_step}")
print(f"  Tempo: {train_result.metrics.get('train_runtime', 0) / 60:.1f} min")

In [ ]:
# Avaliar
eval_result = trainer.evaluate()
print(f"Eval loss: {eval_result['eval_loss']:.4f}")
print(f"Perplexity: {torch.exp(torch.tensor(eval_result['eval_loss'])):.2f}")

## 7. Salvar Adapter LoRA no Drive

Salva apenas o adapter (~16 MB) ao inves do modelo inteiro (~3.4 GB).
Na hora de carregar, combinamos o adapter com o modelo base.

In [ ]:
print(f"Salvando adapter LoRA em: {ADAPTER_SAVE_DIR}")
os.makedirs(ADAPTER_SAVE_DIR, exist_ok=True)

# Salvar apenas o adapter (muito menor que o modelo completo)
model.save_pretrained(ADAPTER_SAVE_DIR)
tokenizer.save_pretrained(ADAPTER_SAVE_DIR)

# Salvar metadados
metadata = {
    'base_model': BASE_MODEL,
    'total_songs': len(songs),
    'genres': dict(genre_counts),
    'max_length': MAX_LENGTH,
    'lora_r': lora_config.r,
    'lora_alpha': lora_config.lora_alpha,
    'epochs': int(training_args.num_train_epochs),
    'final_loss': train_result.training_loss,
    'eval_loss': eval_result['eval_loss'],
}

with open(os.path.join(ADAPTER_SAVE_DIR, 'training_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Tamanho
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_SAVE_DIR, f))
    for f in os.listdir(ADAPTER_SAVE_DIR)
    if os.path.isfile(os.path.join(ADAPTER_SAVE_DIR, f))
)
print(f"\nAdapter salvo! Tamanho: {total_size / (1024**2):.1f} MB")
print("(Apenas o adapter, nao o modelo base de 3.4 GB)")

---

## 8. Carregar Modelo do Drive (Pular Treino)

**Use esta secao nas proximas sessoes.**

Prerequisito: execute as secoes 1 (Setup) e 2 (Montar Drive) antes.

In [ ]:
# ====================================================================
# CARREGAR MODELO JA TREINADO
# Use esta celula ao inves de treinar novamente.
# ====================================================================

import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DRIVE_BASE = '/content/drive/MyDrive/musicas_model'
ADAPTER_SAVE_DIR = os.path.join(DRIVE_BASE, 'bloom-letras-lora')
BASE_MODEL = 'bigscience/bloom-1b7'

if not os.path.exists(ADAPTER_SAVE_DIR):
    print(f"Adapter nao encontrado em {ADAPTER_SAVE_DIR}")
    print("Voce precisa treinar primeiro (secoes 3-7).")
else:
    print(f"Carregando modelo base {BASE_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_SAVE_DIR)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map='auto',
    )

    print("Carregando adapter LoRA...")
    model = PeftModel.from_pretrained(model, ADAPTER_SAVE_DIR)
    model.eval()

    # Metadados
    meta_path = os.path.join(ADAPTER_SAVE_DIR, 'training_metadata.json')
    if os.path.exists(meta_path):
        with open(meta_path, 'r') as f:
            metadata = json.load(f)
        print(f"\nModelo carregado!")
        print(f"  Base: {metadata.get('base_model', '?')}")
        print(f"  Musicas: {metadata.get('total_songs', '?')}")
        print(f"  LoRA r={metadata.get('lora_r', '?')}, alpha={metadata.get('lora_alpha', '?')}")
        print(f"  Loss: {metadata.get('final_loss', '?'):.4f}")
    else:
        print("\nModelo carregado!")

    print(f"Dispositivo: {device}")
    print("Pronto para gerar letras!")

---

## 9. Gerar Letras

### Parametros de Geracao

| Parametro | Descricao | Padrao |
|-----------|-----------|--------|
| `genero` | Genero musical | `'Pagode'` |
| `inicio` | Primeiras palavras (opcional) | `''` |
| `temperature` | Criatividade (0.3=seguro, 1.2=ousado) | `0.8` |
| `top_k` | Top-K sampling | `50` |
| `top_p` | Nucleus sampling | `0.92` |
| `max_tokens` | Comprimento maximo | `300` |
| `num_letras` | Quantas gerar | `1` |

In [ ]:
def limpar_letra(texto):
    """
    Pos-processamento: limpa a saida do modelo.
    Remove artefatos, linhas incompletas, lixo.
    """
    # Extrair apenas a parte da letra
    if '[FIM]' in texto:
        texto = texto.split('[FIM]')[0]

    # Remover o header de genero se presente
    lines = texto.strip().split('\n')
    cleaned = []
    skip_header = True

    for line in lines:
        stripped = line.strip()

        # Pular header [Genero: ...]
        if skip_header and (stripped.startswith('[Genero:') or stripped == ''):
            if stripped.startswith('[Genero:'):
                skip_header = False
            continue
        skip_header = False

        # Parar se encontrar outro header de genero (modelo gerou outra musica)
        if stripped.startswith('[Genero:'):
            break

        # Remover linhas com lixo comum
        if stripped.startswith('###') or stripped.startswith('==='):
            break

        cleaned.append(line)

    # Remover ultima linha se parece incompleta (nao termina com vogal/pontuacao)
    if cleaned:
        last = cleaned[-1].strip()
        if last and len(last) < 10 and not last[-1] in 'aeiouãõéêíóúàAEIOUr.!?,)"':
            cleaned = cleaned[:-1]

    # Remover linhas em branco consecutivas (max 1)
    result = []
    prev_blank = False
    for line in cleaned:
        is_blank = line.strip() == ''
        if is_blank and prev_blank:
            continue
        result.append(line)
        prev_blank = is_blank

    # Remover linhas em branco no inicio e fim
    text = '\n'.join(result).strip()
    return text


def gerar_letra(
    genero='Pagode',
    inicio='',
    temperature=0.8,
    top_k=50,
    top_p=0.92,
    max_tokens=300,
    repetition_penalty=1.2,
    num_letras=1,
):
    """
    Gera letras condicionadas por genero.

    Args:
        genero: 'MPB', 'Trap', 'Sertanejo', 'Pagode', 'Arrocha'
        inicio: Primeiras palavras/linhas (opcional)
        temperature: Criatividade (0.3 a 1.3)
        top_k, top_p: Sampling
        max_tokens: Tokens maximos
        repetition_penalty: Penalidade por repeticao
        num_letras: Quantas gerar

    Returns:
        Lista de letras geradas
    """
    # Prompt em linguagem natural
    prompt = f"[Genero: {genero}]\n\n"
    if inicio:
        prompt += inicio

    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    resultados = []
    for _ in range(num_letras):
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens=max_tokens,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                do_sample=True,
                num_return_sequences=1,
                pad_token_id=tokenizer.pad_token_id,
            )

        texto = tokenizer.decode(output[0], skip_special_tokens=True)
        letra = limpar_letra(texto)

        if letra:  # So adicionar se nao ficou vazio
            resultados.append(letra)

    return resultados


def exibir_letra(letra, genero='', titulo='', numero=None):
    """Exibe uma letra formatada."""
    print("\n" + "=" * 60)
    header = ""
    if numero is not None:
        header += f"#{numero} "
    if genero:
        header += f"[{genero}]"
    if titulo:
        header += f" - {titulo}"
    print(f" {header} ".center(60, "="))
    print("=" * 60)
    print()
    print(letra)
    print()
    palavras = len(letra.split())
    linhas = len([l for l in letra.splitlines() if l.strip()])
    print(f"--- {palavras} palavras, {linhas} linhas ---")
    print()


print("Funcoes de geracao prontas.")
print("Generos: MPB, Trap, Sertanejo, Pagode, Arrocha")

### 9.1 Gerar por Genero

In [ ]:
# Escolha o genero e ajuste os parametros
GENERO = 'Pagode'
TEMPERATURA = 0.8
NUM_LETRAS = 3

letras = gerar_letra(
    genero=GENERO,
    temperature=TEMPERATURA,
    num_letras=NUM_LETRAS,
)

for i, letra in enumerate(letras, 1):
    exibir_letra(letra, genero=GENERO, numero=i)

### 9.2 Gerar com Inicio da Letra

In [ ]:
# Direcionar o tema com as primeiras linhas
letras = gerar_letra(
    genero='MPB',
    inicio='Da janela eu vejo o mar\nE o mar me leva pra longe\n',
    temperature=0.85,
    num_letras=2,
)

for i, letra in enumerate(letras, 1):
    exibir_letra(letra, genero='MPB', numero=i)

### 9.3 Comparar Generos (mesmo inicio)

In [ ]:
# Mesmo inicio, generos diferentes
INICIO = 'Hoje eu acordei pensando em voce\n'

print("Mesmo inicio em todos os generos:\n")

for genero in ['MPB', 'Pagode', 'Sertanejo', 'Arrocha', 'Trap']:
    letras = gerar_letra(
        genero=genero,
        inicio=INICIO,
        temperature=0.8,
        num_letras=1,
    )
    if letras:
        exibir_letra(letras[0], genero=genero)

### 9.4 Explorar Temperaturas

In [ ]:
GENERO_TESTE = 'Sertanejo'

print(f"Temperaturas diferentes para {GENERO_TESTE}:\n")

for temp in [0.4, 0.7, 1.0, 1.3]:
    letras = gerar_letra(
        genero=GENERO_TESTE,
        temperature=temp,
        max_tokens=150,
        num_letras=1,
    )
    if letras:
        print(f"--- Temperatura {temp} ---")
        for line in letras[0].splitlines()[:6]:
            print(f"  {line}")
        print()

## 10. Gerador Interativo

In [ ]:
# @title Gerador Interativo { run: "auto", form-width: "80%" }

genero = 'Pagode'  # @param ['MPB', 'Trap', 'Sertanejo', 'Pagode', 'Arrocha']
inicio_da_letra = ''  # @param {type: "string"}
temperatura = 0.8  # @param {type: "slider", min: 0.3, max: 1.3, step: 0.1}
top_k = 50  # @param {type: "slider", min: 10, max: 100, step: 10}
top_p = 0.92  # @param {type: "slider", min: 0.5, max: 1.0, step: 0.02}
penalidade_repeticao = 1.2  # @param {type: "slider", min: 1.0, max: 2.0, step: 0.1}
max_tokens = 300  # @param {type: "slider", min: 100, max: 500, step: 50}
quantidade = 1  # @param {type: "slider", min: 1, max: 5, step: 1}

letras = gerar_letra(
    genero=genero,
    inicio=inicio_da_letra,
    temperature=temperatura,
    top_k=top_k,
    top_p=top_p,
    max_tokens=max_tokens,
    repetition_penalty=penalidade_repeticao,
    num_letras=quantidade,
)

for i, letra in enumerate(letras, 1):
    exibir_letra(letra, genero=genero, numero=i)

## 11. Exportar Letras

In [ ]:
def salvar_letras(letras, genero, filepath=None):
    """Salva letras geradas no Drive."""
    if filepath is None:
        output_dir = os.path.join(DRIVE_BASE, 'letras_geradas')
        os.makedirs(output_dir, exist_ok=True)
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filepath = os.path.join(output_dir, f"{genero}_{timestamp}.txt")

    with open(filepath, 'w', encoding='utf-8') as f:
        for i, letra in enumerate(letras, 1):
            f.write(f"### Letra #{i} [{genero}]\n\n")
            f.write(letra)
            f.write("\n\n" + "=" * 60 + "\n\n")

    print(f"Salvo em: {filepath}")
    return filepath


# Exemplo: salvar_letras(letras, genero='Pagode')

## 12. Dicas

### Temperatura
- **0.3-0.5**: Conservador, fiel ao corpus. Bom para Sertanejo/Arrocha.
- **0.7-0.9**: Equilibrado. Recomendado para a maioria dos casos.
- **1.0-1.3**: Criativo/experimental. Bom para MPB/Trap.

### Penalidade de Repeticao
- **1.0**: Permite repeticoes (refroes naturais)
- **1.2**: Leve (recomendado)
- **1.5+**: Forte (forca diversidade, pode perder naturalidade)

### Workflow
1. Gere 3-5 variantes (`num_letras=3`)
2. Escolha trechos bons de cada
3. Use `inicio` para regenerar partes especificas
4. Combine e edite manualmente

### Generos
- **MPB**: Poetico, sofisticado (~179 palavras/musica)
- **Trap**: Longo, mix de estilos (~418 palavras/musica)
- **Sertanejo**: Amor/sofrencia, refroes (~240 palavras/musica)
- **Pagode**: Romance, emocao (~232 palavras/musica)
- **Arrocha**: Direto, emotivo (~182 palavras/musica)

---

Modelo: BLOOM-1b7 (1.7B params) + LoRA | Corpus: 5.159 musicas | 5 generos